In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(
    '/content/Comment Toxicity Detection train.csv',
    on_bad_lines='skip'
)

In [3]:
one = df.copy()

In [4]:
one.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [5]:
one.isnull().sum()

,0
id,0
comment_text,0
toxic,0
severe_toxic,0
obscene,0
threat,0
insult,0
identity_hate,0


In [6]:
one.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [7]:
import re

def clean_text(text):

    # lowercase
    text = text.lower()

    # remove newline
    text = text.replace('\n', ' ')

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text
one['comment_text'] =one['comment_text'].apply(clean_text)

In [8]:
one.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,explanation why the edits made under my userna...,0,0,0,0,0,0
1,000103f0d9cfb60f,daww he matches this background colour im seem...,0,0,0,0,0,0
2,000113f07ec002fd,hey man im really not trying to edit war its j...,0,0,0,0,0,0
3,0001b41b1c6bb37e,more i cant make any real suggestions on impro...,0,0,0,0,0,0
4,0001d958c54c6e35,you sir are my hero any chance you remember wh...,0,0,0,0,0,0


In [9]:
one['comment_text'].unique()

array(['explanation why the edits made under my username hardcore metallica fan were reverted they werent vandalisms just closure on some gas after i voted at new york dolls fac and please dont remove the template from the talk page since im retired now',
       'daww he matches this background colour im seemingly stuck with thanks talk january utc',
       'hey man im really not trying to edit war its just that this guy is constantly removing relevant information and talking to me through edits instead of my talk page he seems to care more about the formatting than the actual info',
       ...,
       'spitzer umm theres no actual article for prostitution ring crunch captain',
       'and it looks like it was actually you who put on the speedy to have the first version deleted now that i look at it',
       'and i really dont think you understand i came here and my idea was bad right away what kind of community goes you have bad ideas go away instead of helping rewrite them'],
      d

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [11]:
tokenizer = Tokenizer(num_words=5000)

In [12]:
X = one['comment_text']

y = one[
    [
        'toxic',
        'severe_toxic',
        'obscene',
        'threat',
        'insult',
        'identity_hate'
    ]
]

In [13]:
tokenizer.fit_on_texts(X)

In [14]:
X_seq = tokenizer.texts_to_sequences(X)
X[0]

X_seq[0]

[638,
 75,
 1,
 122,
 126,
 173,
 28,
 628,
 4522,
 1040,
 82,
 311,
 52,
 2010,
 50,
 15,
 61,
 2605,
 143,
 7,
 2759,
 33,
 114,
 1131,
 2791,
 4,
 45,
 54,
 234,
 1,
 409,
 30,
 1,
 41,
 27,
 141,
 69,
 3337,
 89]

In [15]:
lengths = [len(seq) for seq in X_seq]
print(max(lengths))
print(min(lengths))

1399
0


In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [17]:
import numpy as np


lengths = [len(seq) for seq in X_seq]

print(np.percentile(lengths,90))

print(np.percentile(lengths,95))

print(np.percentile(lengths,99))

134.0
205.0
489.0


In [18]:

X_pad = pad_sequences(X_seq, maxlen=200)

print(X_pad.shape)

(159571, 200)


In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(127656, 200)
(31915, 200)
(127656, 6)
(31915, 6)


In [21]:
import torch

X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

In [22]:
print(X_train.shape)
print(y_train.shape)

torch.Size([127656, 200])
torch.Size([127656, 6])


In [23]:
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train, y_train)

test_dataset = TensorDataset(X_test, y_test)

In [24]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [25]:
import torch.nn as nn


In [26]:

len(tokenizer.word_index)

223959

In [107]:
class ToxicityLSTM(nn.Module):

    def __init__(self):

        super(ToxicityLSTM, self).__init__()

        # embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=5000,
            embedding_dim=128
        )

        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            batch_first=True
        )

        # dropout layer
        self.dropout = nn.Dropout(0.5)

        # fully connected output layer
        self.fc = nn.Linear(64, 6)

    def forward(self, x):

        # convert word indexes into vectors
        x = self.embedding(x)

        # pass vectors into LSTM
        lstm_out, (hidden, cell) = self.lstm(x)

        # take final hidden state
        x = hidden[-1]

        # apply dropout
        x = self.dropout(x)

        # final prediction
        x = self.fc(x)

        return x


# create model
model = ToxicityLSTM()

# loss function
criterion = nn.BCEWithLogitsLoss()

# optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# device setup
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# move model to device
model = model.to(device)

print(device)

# training
epochs = 25

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs, labels in train_loader:

        # move data to device
        inputs = inputs.to(device)
        labels = labels.to(device)

        # clear old gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(inputs)

        # calculate loss
        loss = criterion(outputs, labels)

        # backpropagation
        loss.backward()

        # update weights
        optimizer.step()

        total_loss += loss.item()
    print(f"loss{total_loss}")
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


cuda
loss336.6517778080888
Epoch 1, Loss: 336.6518
loss221.24273765925318
Epoch 2, Loss: 221.2427
loss204.2398128916975
Epoch 3, Loss: 204.2398
loss190.26162495242897
Epoch 4, Loss: 190.2616
loss180.68096086813603
Epoch 5, Loss: 180.6810
loss170.83525700960308
Epoch 6, Loss: 170.8353
loss161.98677762434818
Epoch 7, Loss: 161.9868
loss153.37439310492482
Epoch 8, Loss: 153.3744
loss145.76215667102952
Epoch 9, Loss: 145.7622
loss138.24932571777026
Epoch 10, Loss: 138.2493
loss132.74060159691726
Epoch 11, Loss: 132.7406
loss126.8323472212578
Epoch 12, Loss: 126.8323
loss121.68264725900372
Epoch 13, Loss: 121.6826
loss116.91518464806722
Epoch 14, Loss: 116.9152
loss112.78177300244715
Epoch 15, Loss: 112.7818
loss108.48543752924161
Epoch 16, Loss: 108.4854
loss104.82899269441259
Epoch 17, Loss: 104.8290
loss103.38668610547029
Epoch 18, Loss: 103.3867
loss99.97988998072105
Epoch 19, Loss: 99.9799
loss96.76964663998297
Epoch 20, Loss: 96.7696
loss95.72908690338227
Epoch 21, Loss: 95.7291
loss9

In [108]:
# evaluation mode
model.eval()

# store total test loss
test_loss = 0

# disable gradient calculation
with torch.no_grad():

    for inputs, labels in test_loader:

        # move data to device
        inputs = inputs.to(device)
        labels = labels.to(device)

        # forward pass
        outputs = model(inputs)

        # calculate loss
        loss = criterion(outputs, labels)

        # add batch loss
        test_loss += loss.item()

# print final test loss
print("Test Loss:", test_loss)

Test Loss: 81.73375324268818


In [109]:
sample_output = torch.sigmoid(outputs)

print(sample_output[0])

tensor([9.9698e-01, 5.2280e-03, 4.1223e-01, 6.5677e-06, 7.4783e-01, 2.2281e-02],
       device='cuda:0')


In [110]:
predictions = (sample_output > 0.5).float()

print(predictions[0])

tensor([1., 0., 0., 0., 1., 0.], device='cuda:0')


In [111]:
avg_test_loss = test_loss / len(test_loader)

print(avg_test_loss)

0.08189754833936691


In [112]:
torch.save(
    model.state_dict(),
    "toxicity_model.pth"
)

In [113]:
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [114]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import pickle
import re
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------------
# Text Cleaning Function
# -----------------------------------
def clean_text(text):
    text = str(text).lower()
    text = text.replace('\n', ' ')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# -----------------------------------
# LSTM Model
# -----------------------------------
class ToxicityLSTM(nn.Module):
    def __init__(self):
        super(ToxicityLSTM, self).__init__()
        self.embedding = nn.Embedding(5000, 128)
        self.lstm = nn.LSTM(128, 64, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(64, 6)

    def forward(self, x):
        x = self.embedding(x)
        output, (hidden, cell) = self.lstm(x)
        x = hidden[-1]
        x = self.dropout(x)
        x = self.fc(x)
        return x

# -----------------------------------
# Load Assets
# -----------------------------------
@st.cache_resource
def load_assets():
    with open("tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)
    model = ToxicityLSTM()
    model.load_state_dict(torch.load("toxicity_model.pth", map_location=torch.device("cpu")))
    model.eval()
    return tokenizer, model

tokenizer, model = load_assets()
labels = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

st.title("Comment Toxicity Detector")

# --- Single Prediction ---
st.header("Single Comment Prediction")
comment = st.text_area("Enter Comment")

if st.button("Analyze"):
    if comment.strip():
        cleaned = clean_text(comment)
        seq = tokenizer.texts_to_sequences([cleaned])
        padded = pad_sequences(seq, maxlen=200)
        input_t = torch.tensor(padded, dtype=torch.long)
        with torch.no_grad():
            out = model(input_t)
            probs = torch.sigmoid(out).numpy()[0]

        st.subheader("Results")
        cols = st.columns(3)
        for i, label in enumerate(labels):
            with cols[i % 3]:
                st.metric(label, f"{probs[i]:.2%}")
                st.progress(float(probs[i]))
    else:
        st.warning("Please enter text.")

# --- Bulk Prediction ---
st.header("Bulk CSV Prediction")
uploaded_file = st.file_uploader("Upload CSV", type=["csv"])
if uploaded_file:
    df = pd.read_csv(uploaded_file)
    if "comment_text" not in df.columns:
        st.error("CSV must have a 'comment_text' column.")
    else:
        with st.spinner('Analyzing...'):
            all_probs = []
            for text in df["comment_text"]:
                cleaned = clean_text(text)
                seq = tokenizer.texts_to_sequences([cleaned])
                padded = pad_sequences(seq, maxlen=200)
                input_t = torch.tensor(padded, dtype=torch.long)
                with torch.no_grad():
                    out = model(input_t)
                    probs = torch.sigmoid(out).numpy()[0]
                    all_probs.append(probs)

            prob_array = np.array(all_probs)
            for i, label in enumerate(labels):
                df[label] = (prob_array[:, i] > 0.5).astype(int)

            st.success("Analysis Complete!")
            st.dataframe(df.head())
            st.download_button("Download CSV with Predictions", df.to_csv(index=False), "predictions.csv", "text/csv")

Overwriting app.py


In [115]:
!pip install pyngrok streamlit
from pyngrok import ngrok

In [116]:
import os
import time
# Restart the Streamlit application using the python module and nohup
# This ensures the process continues and finds the streamlit package
os.system('nohup python -m streamlit run app.py --server.port 8501 > /content/logs.txt 2>&1 &')
time.sleep(5)
print('Streamlit server restart command sent. Checking logs...')
!tail /content/logs.txt

Streamlit server restart command sent. Checking logs...




In [117]:
# STEP 5 ------------------------------------
# Add ngrok Auth Token

ngrok.set_auth_token("3D1gKUjT4FxghAQMeM2toSrvny8_2E17xUa9cnKs2bSy8Uktp")

In [118]:
# Kill any existing tunnels to avoid ERR_NGROK_324
ngrok.kill()

# Connect to the Streamlit port
public_url = ngrok.connect(8501)
print(f"Streamlit is being exposed at: {public_url}")

Streamlit is being exposed at: NgrokTunnel: "https://grouped-unnerving-handshake.ngrok-free.dev" -> "http://localhost:8501"


In [119]:
!cat /content/logs.txt



2026-05-17 11:03:57.365 Port 8501 is not available
